# 01 – Data Collection and Preparation

## Context
This notebook is the first step of a project on the effect of diversification in a portfolio.
It builds the clean price dataset used by all the following notebooks.

## Data
- **Source:** Yahoo Finance, via `yfinance`
- **Assets:** GLD, ASML.AS, TTE.PA, AI.PA, MC.PA, IWDA.AS and BTC-USD, covering gold, cryptocurrency, European equities from different sectors and a global equity ETF
- **Period:** rolling 5 years, long enough to cover several market phases
- **Prices:** adjusted closing prices
- **Base currency:** USD, using daily exchange rates from Yahoo Finance
- **Dates kept:** only the days when all assets and exchange rates have a price
- **Output:** `Data/Processed/dataset_clean.csv`

## Contents
1. Identify each asset's currency
2. Build the required exchange rate tickers
3. Download prices and exchange rates
4. Clean the dataset
5. Convert prices to the base currency
6. Export the clean dataset

## Setup

In [50]:
import yfinance as yf
from datetime import date, timedelta
from pathlib import Path

import warnings
warnings.filterwarnings("ignore", message=".*inplace keyword.*")

In [51]:
TICKERS = ["GLD","ASML.AS","TTE.PA","AI.PA","MC.PA","IWDA.AS","BTC-USD","NVDA","EXS1.DE"]
BASE_CURRENCY = "USD"
N_YEARS =5
START = date.today() - timedelta(days=N_YEARS*365)
END = date.today()
PROCESSED_DIR = Path("../Data/Processed")

## Asset currencies

In [52]:
# --- Get the currency of each asset ---
currency_by_ticker = {}
for ticker in TICKERS :
    currency_by_ticker[ticker] = yf.Ticker(ticker).fast_info['currency']

print(f"\nCurrency of each asset: {currency_by_ticker} \n")


Currency of each asset: {'GLD': 'USD', 'ASML.AS': 'EUR', 'TTE.PA': 'EUR', 'AI.PA': 'EUR', 'MC.PA': 'EUR', 'IWDA.AS': 'EUR', 'BTC-USD': 'USD', 'NVDA': 'USD', 'EXS1.DE': 'EUR'} 



## Exchange rate tickers

In [53]:
# --- Build the exchange rate tickers ---

currencies_to_convert= set(currency_by_ticker.values()) # unique currencies
currencies_to_convert.discard(BASE_CURRENCY)# remove the base currency (does nothing if it is absent)
print(currencies_to_convert)

# --- Map each currency to its exchange rate ticker ---
fx_ticker_by_currency = {}

for currency in currencies_to_convert :
    fx_ticker_by_currency[currency] = f"{currency}{BASE_CURRENCY}=X" #"EUR" → "EURUSD=X".
print(fx_ticker_by_currency)

fx_tickers = list(fx_ticker_by_currency.values())
print(f"FX tickers : {fx_tickers}")

{'EUR'}
{'EUR': 'EURUSD=X'}
FX tickers : ['EURUSD=X']


## Download and cleaning
Adjusted prices are used because they account for dividends and stock splits, which gives a more accurate measure of returns.

To allow an efficient comparison between assets, I decided to drop the rows where a missing value is detected (see the row counts printed below).
Missing values come from different market calendars: for example, Bitcoin trades 7 days a week, while stock exchanges close on weekends and public holidays. Filling these gaps would create artificial days with a 0% return and distort volatility.

In [54]:
# --- Download the full dataset ---
raw_all = yf.download(TICKERS + fx_tickers, start=START, end=END, auto_adjust=True)
raw_close = raw_all['Close']
print(raw_close.isna().sum())
print(raw_close.shape)
raw_close = raw_close.reset_index().dropna()
print(raw_close.shape)

[*********************100%***********************]  10 of 10 completed

Ticker
AI.PA       545
ASML.AS     545
BTC-USD       0
EURUSD=X    525
EXS1.DE     552
GLD         570
IWDA.AS     546
MC.PA       545
NVDA        570
TTE.PA      545
dtype: int64
(1825, 10)
(1235, 11)


## Currency conversion
All prices are converted to USD so that the assets are compared in a single currency. Returns therefore include the EUR/USD effect, as they would for a USD-based investor.

In [55]:
# --- Convert the columns to the base currency ---
dataset_clean = raw_close.copy()

column_to_convert = {}
for ticker,currency in currency_by_ticker.items() :
    if currency in fx_ticker_by_currency: 
        column_to_convert[ticker] = fx_ticker_by_currency[currency]

print(f"Columns to convert: {column_to_convert}")


for ticker_to_convert,currency_change in column_to_convert.items():
    dataset_clean[ticker_to_convert] = dataset_clean[ticker_to_convert] * dataset_clean[currency_change]
            
dataset_clean.head()

dataset_clean = dataset_clean.drop(columns=fx_tickers)




dataset_clean.head()

Columns to convert: {'ASML.AS': 'EURUSD=X', 'TTE.PA': 'EURUSD=X', 'AI.PA': 'EURUSD=X', 'MC.PA': 'EURUSD=X', 'IWDA.AS': 'EURUSD=X', 'EXS1.DE': 'EURUSD=X'}


Ticker,Date,AI.PA,ASML.AS,BTC-USD,EXS1.DE,GLD,IWDA.AS,MC.PA,NVDA,TTE.PA
0,2021-11-15,119.085870,822.379345,63557.871094,156.659042,174.179993,89.319228,748.550547,29.893333,37.603724
1,2021-11-16,117.587358,819.038976,60161.246094,156.698269,172.919998,89.476918,752.814672,30.070551,37.544998
2,2021-11-17,117.599472,827.668320,60368.011719,156.051172,174.500000,89.086386,746.115247,29.132681,37.122529
3,2021-11-18,117.847572,833.074351,56942.136719,155.847399,173.940002,88.758071,748.581460,31.536095,36.721820
4,2021-11-19,119.507049,825.041304,58119.578125,155.860981,172.610001,89.452373,756.863979,32.840355,35.869030


## Verification
The pipeline was checked by hand before relying on it:
- **Different portfolios:** the notebook was run with several sets of tickers, and converted prices were recomputed by hand (e.g. ASML.AS × EURUSD) and matched the output.
- **Other currency:** a Chinese stock quoted in CNY (`600519.SS`) was added as a test. The `CNYUSD=X` ticker was generated automatically and the converted price (Moutai × CNYUSD) matched the hand calculation. It was then removed from the final asset list.
- **Side effect observed:** adding a market with its own holidays (Chinese New Year, Golden Week) removes about 80 additional common trading days after `dropna()`.

Known limits: markets and exchange rates do not close at the same time, and public holidays differ between markets.

## Export

In [56]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
dataset_clean.to_csv(PROCESSED_DIR / "dataset_clean.csv", index=False)